In [1]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00


In [2]:
%%writefile categories.py
"""
Schema 5 category cho VinFast callbot.

Triết lý: đây là SINGLE SOURCE OF TRUTH. State machine, LLM prompt, và eval
đều đọc từ đây. Sửa field ở 1 chỗ -> cả hệ thống cập nhật.

Mỗi Field mô tả: cách hỏi, có bắt buộc không, có cần đọc-lại-xác-nhận không
(field nhạy cảm như số đt/biển số), và regex validate sau khi normalize.
"""
from dataclasses import dataclass, field as dc_field
from typing import Optional
import re


@dataclass
class Field:
    key: str                       # tên field trong JSON output
    label: str                     # nhãn tiếng Việt (cho prompt/log)
    ask: str                       # câu hỏi gợi ý khi cần thu field này
    required: bool = True
    verify: bool = False           # field nhạy cảm -> đọc lại xác nhận khi nghi ngờ (xử lý ca "garbled input")
    pattern: Optional[str] = None  # regex validate sau normalize (None = bỏ qua)
    enum: Optional[list] = None    # tập giá trị hợp lệ (None = free text)

    def is_valid(self, value: str) -> bool:
        """Validate giá trị đã normalize. Dùng để bắt input rác trước khi ghi nhận."""
        if value is None or str(value).strip() == "":
            return False
        v = str(value).strip()
        if self.enum and v.lower() not in [e.lower() for e in self.enum]:
            return False
        if self.pattern and not re.match(self.pattern, v):
            return False
        return True


# ---- Validators dùng chung ----
# SĐT VN: bắt đầu 0 hoặc +84, tổng 9-11 chữ số (loose, đủ cho demo)
PHONE = r"^(?:\+?84|0)\d{8,10}$"
# Odo: số nguyên (km)
ODO = r"^\d{1,7}$"
# Biển số/VIN: để verify=True và check loose vì format rất đa dạng
#   (ô tô 30A-123.45, xe máy 29X1-12345, VIN 17 ký tự alnum)
PLATE_VIN = r"^[A-Z0-9][A-Z0-9.\-\s]{4,20}$"


def F(key, label, ask, **kw):
    return Field(key=key, label=label, ask=ask, **kw)


CATEGORIES = {
    "G_1": {
        "name": "Cứu hộ (Roadside Rescue)",
        "goal": "Nhận báo hỏng xe và điều phối hỗ trợ (điều xe dịch vụ hoặc cứu hộ kéo xe)",
        "can_emergency": True,                       # category có thể phát sinh tình huống khẩn cấp
        "critical_fields": ["phone", "current_location", "city_name"],  # field tối thiểu khi emergency
        "fields": [
            F("full_name", "Họ tên", "Cho em xin họ tên của anh/chị ạ?"),
            F("phone", "Số điện thoại", "Anh/chị cho em xin số điện thoại liên hệ ạ?", verify=True, pattern=PHONE),
            F("vehicle_model", "Dòng/model xe", "Xe của mình là dòng nào ạ (VF8, VF9, ...)?"),
            F("license_plate_vin", "Biển số/VIN", "Anh/chị đọc giúp em biển số xe (hoặc số VIN) ạ?", verify=True, pattern=PLATE_VIN),
            F("vehicle_type", "Loại xe", "Xe của mình là ô tô điện hay loại nào ạ?"),
            F("current_odo", "Số odo hiện tại", "Số odo (số km đã đi) hiện tại khoảng bao nhiêu ạ?", pattern=ODO),
            F("current_location", "Vị trí hiện tại", "Hiện tại xe đang ở vị trí nào ạ? Anh/chị mô tả giúp em cụ thể nhất có thể."),
            F("city_name", "Tỉnh/thành", "Mình đang ở tỉnh/thành phố nào ạ?"),
            F("vehicle_condition", "Tình trạng xe", "Xe đang gặp tình trạng gì ạ? Anh/chị mô tả giúp em."),
        ],
    },
    "G_2": {
        "name": "Bảo hành & Sửa chữa (Warranty & Repair)",
        "goal": "Tư vấn chính sách bảo hành/sửa chữa và cung cấp thông tin trung tâm dịch vụ để đặt lịch",
        "can_emergency": False,
        "critical_fields": [],
        "fields": [
            F("full_name", "Họ tên", "Cho em xin họ tên của anh/chị ạ?"),
            F("owner_phone", "SĐT chủ xe", "Anh/chị cho em xin số điện thoại chủ xe ạ?", verify=True, pattern=PHONE),
            F("vehicle_model", "Dòng/model xe", "Xe của mình là dòng nào ạ?"),
            F("vehicle_usage_type", "Mục đích sử dụng", "Xe dùng cho mục đích cá nhân hay kinh doanh/dịch vụ ạ?"),
            F("license_plate_vin", "Biển số/VIN", "Anh/chị đọc giúp em biển số xe (hoặc số VIN) ạ?", verify=True, pattern=PLATE_VIN),
            F("service_center", "Trung tâm dịch vụ", "Mình muốn tới trung tâm dịch vụ ở khu vực nào ạ?"),
            F("vehicle_condition", "Tình trạng xe", "Xe đang gặp vấn đề gì cần bảo hành/sửa chữa ạ?"),
        ],
    },
    "G_3": {
        "name": "Đơn hàng (Order Status & Management)",
        "goal": "Tư vấn tình trạng đơn & quy trình giao xe; xử lý chuyển cọc hoặc cập nhật thông tin đơn",
        "can_emergency": False,
        "critical_fields": [],
        "fields": [
            F("full_name", "Họ tên", "Cho em xin họ tên của anh/chị ạ?"),
            F("order_phone", "SĐT đặt hàng", "Anh/chị cho em xin số điện thoại đã dùng để đặt hàng ạ?", verify=True, pattern=PHONE),
            F("order_code_dealer", "Mã đơn/đại lý", "Anh/chị đọc giúp em mã đơn hàng (hoặc tên đại lý) ạ?", verify=True),
            F("customer_type", "Loại khách hàng", "Mình là khách hàng cá nhân hay doanh nghiệp ạ?", enum=["cá nhân", "doanh nghiệp"]),
        ],
    },
    "G_4": {
        "name": "Xe máy – Bảo hành (Motorbike Warranty)",
        "goal": "Tư vấn chính sách bảo hành/sửa chữa xe máy điện và chỉ tới trung tâm dịch vụ gần nhất",
        "can_emergency": False,
        "critical_fields": [],
        "fields": [
            F("full_name", "Họ tên", "Cho em xin họ tên của anh/chị ạ?"),
            F("phone", "Số điện thoại", "Anh/chị cho em xin số điện thoại liên hệ ạ?", verify=True, pattern=PHONE),
            F("vehicle_line", "Dòng xe máy", "Xe máy điện của mình là dòng nào ạ (Klara, Vento, Feliz, ...)?"),
            F("license_plate_vin", "Biển số/VIN", "Anh/chị đọc giúp em biển số xe (hoặc số khung) ạ?", verify=True, pattern=PLATE_VIN),
            F("current_location", "Vị trí hiện tại", "Mình đang ở khu vực nào ạ? Để em tìm trung tâm gần nhất."),
            F("vehicle_condition", "Tình trạng xe", "Xe đang gặp vấn đề gì ạ?"),
        ],
    },
    "G_5": {
        "name": "Hỗ trợ kỹ thuật từ xa (Remote Tech Support)",
        "goal": "Hướng dẫn khắc phục từ xa; đặt lịch tới trung tâm nếu không xử lý được",
        "can_emergency": False,
        "critical_fields": [],
        "fields": [
            F("full_name", "Họ tên", "Cho em xin họ tên của anh/chị ạ?"),
            F("phone", "Số điện thoại", "Anh/chị cho em xin số điện thoại liên hệ ạ?", verify=True, pattern=PHONE),
            F("license_plate_vin", "Biển số/VIN", "Anh/chị đọc giúp em biển số xe (hoặc số VIN) ạ?", verify=True, pattern=PLATE_VIN),
            F("vehicle_line", "Dòng xe", "Xe của mình là dòng nào ạ?"),
            F("current_odo", "Số odo", "Số odo hiện tại khoảng bao nhiêu ạ? (không bắt buộc)", required=False, pattern=ODO),
            F("vehicle_condition_details", "Chi tiết tình trạng (kèm phiên bản phần mềm)",
              "Anh/chị mô tả chi tiết lỗi giúp em, kèm phiên bản phần mềm nếu biết ạ?"),
        ],
    },
}


# Field post-call (sinh từ transcript cuối cuộc gọi, KHÔNG thu trong hội thoại)
POST_CALL_FIELDS = ["short_summary", "sentimental_analysis", "emergency"]


# ---- Helper truy vấn schema ----
def get_fields(category: str) -> list:
    return CATEGORIES[category]["fields"]

def required_keys(category: str) -> list:
    return [f.key for f in get_fields(category) if f.required]

def get_field(category: str, key: str) -> Optional[Field]:
    return next((f for f in get_fields(category) if f.key == key), None)

def all_categories() -> list:
    return list(CATEGORIES.keys())

Writing categories.py


In [3]:
!head -5 categories.py

"""
Schema 5 category cho VinFast callbot.

Triết lý: đây là SINGLE SOURCE OF TRUTH. State machine, LLM prompt, và eval
đều đọc từ đây. Sửa field ở 1 chỗ -> cả hệ thống cập nhật.


In [4]:
%%writefile dialogue_state.py
"""
State machine = BỘ ĐIỀU KHIỂN xác định của callbot.

Tư tưởng: LLM KHÔNG tự quyết định luồng. LLM chỉ làm 2 việc:
  (1) Trích xuất (extract): từ câu khách nói -> category + field updates + cờ (correction/out_of_scope/emergency/unclear)
  (2) Diễn đạt (render): biến 1 Action xác định -> câu tiếng Việt tự nhiên

State machine cầm lái mọi quyết định luồng -> 8 exception trở nên KIỂM CHỨNG ĐƯỢC,
không phải cầu may vào prompt.

Vòng lặp 1 lượt:
  user_text -> [LLM extract] -> state.apply_extraction(...) -> state.next_action() -> [LLM render] -> reply
"""
from dataclasses import dataclass, field as dc_field
from enum import Enum
from typing import Optional
import categories as cat


MAX_FAIL = 2  # stuck sau 2+ lượt fail -> chuyển human (theo brief)


class Status(str, Enum):
    ACTIVE = "active"
    COMPLETED = "completed"      # đã thu đủ field
    TRANSFERRED = "transferred"  # chuyển human (stuck / out-of-scope / khách yêu cầu)
    HUNG_UP = "hung_up"          # khách cúp máy giữa chừng


class ActionType(str, Enum):
    CLARIFY_INTENT = "clarify_intent"   # ý định mơ hồ -> hỏi 1 câu trước khi route
    ASK_FIELD = "ask_field"             # hỏi field còn thiếu
    CONFIRM_FIELD = "confirm_field"     # đọc lại giá trị nghi ngờ để xác nhận (garbled input)
    EMERGENCY = "emergency"             # ưu tiên cứu hộ, cho hotline, skip field phụ
    OUT_OF_SCOPE = "out_of_scope"       # ngoài phạm vi -> redirect / offer transfer
    TRANSFER_HUMAN = "transfer_human"   # stuck 2+ lượt -> chuyển human
    COMPLETE = "complete"              # đã đủ field -> chốt & tổng kết


@dataclass
class Action:
    type: ActionType
    field_key: Optional[str] = None   # field liên quan (ask/confirm)
    value: Optional[str] = None       # giá trị cần xác nhận (confirm)
    note: Optional[str] = None        # gợi ý thêm cho LLM render


@dataclass
class Slot:
    value: Optional[str] = None
    confirmed: bool = False  # với field verify=True: chỉ tính "có" khi đã confirmed


@dataclass
class DialogueState:
    category: Optional[str] = None
    slots: dict = dc_field(default_factory=dict)          # key -> Slot
    turn_count: int = 0
    fail_count: int = 0                                    # số lượt liên tiếp không tiến triển
    pending_confirm: Optional[str] = None                 # field đang chờ khách xác nhận yes/no
    pending_value: Optional[str] = None
    emergency: bool = False
    status: Status = Status.ACTIVE
    transcript: list = dc_field(default_factory=list)      # [{"role": "user"/"bot", "text": ...}]

    # ---------- ghi nhận lượt thoại ----------
    def log(self, role: str, text: str):
        self.transcript.append({"role": role, "text": text})

    # ---------- cập nhật từ kết quả LLM extract ----------
    def set_category(self, category: str):
        if category in cat.CATEGORIES and self.category != category:
            self.category = category
            for f in cat.get_fields(category):
                self.slots.setdefault(f.key, Slot())

    def apply_updates(self, updates: dict):
        """
        updates: {field_key: value} từ LLM.
        Field verify=True -> KHÔNG confirmed ngay, sẽ phải đọc lại xác nhận.
        Field thường -> validate; hợp lệ thì lưu & confirmed=True.
        Trả về list field cần xác nhận (verify hoặc validate-fail).
        """
        need_confirm = []
        if not self.category:
            return need_confirm
        for key, val in updates.items():
            f = cat.get_field(self.category, key)
            if not f or val is None or str(val).strip() == "":
                continue
            val = str(val).strip()
            slot = self.slots.setdefault(key, Slot())
            if f.verify:
                # field nhạy cảm: lưu tạm, chờ confirm
                slot.value, slot.confirmed = val, False
                need_confirm.append(key)
            elif f.is_valid(val):
                slot.value, slot.confirmed = val, True
            else:
                # input không hợp lệ -> coi như chưa có, sẽ hỏi lại
                need_confirm.append(key)
        return need_confirm

    def confirm(self, key: str, ok: bool):
        """Khách trả lời xác nhận cho field đang pending."""
        slot = self.slots.get(key)
        if not slot:
            return
        if ok:
            slot.confirmed = True
        else:
            slot.value, slot.confirmed = None, False  # sai -> hỏi lại
        if self.pending_confirm == key:
            self.pending_confirm = self.pending_value = None

    def correct(self, key: str, new_value: str):
        """Khách sửa thông tin -> cập nhật, field verify cần confirm lại."""
        f = cat.get_field(self.category, key) if self.category else None
        slot = self.slots.setdefault(key, Slot())
        slot.value = str(new_value).strip()
        slot.confirmed = not (f and f.verify)

    # ---------- bộ đếm fail (stuck detection) ----------
    def register_progress(self):
        self.fail_count = 0

    def register_fail(self):
        self.fail_count += 1

    def mark_hung_up(self):
        self.status = Status.HUNG_UP

    # ---------- truy vấn trạng thái ----------
    def has(self, key: str) -> bool:
        s = self.slots.get(key)
        return bool(s and s.value and s.confirmed)

    def missing_required(self) -> list:
        if not self.category:
            return []
        return [k for k in cat.required_keys(self.category) if not self.has(k)]

    def is_complete(self) -> bool:
        return self.category is not None and not self.missing_required()

    # ---------- TRÁI TIM: quyết định hành động tiếp theo ----------
    def next_action(self) -> Action:
        """
        Quyết định xác định, ưu tiên từ trên xuống. Đây là nơi 8 exception hội tụ.
        LLM chỉ việc render Action này thành câu tiếng Việt.
        """
        # 1. Khẩn cấp: ưu tiên tuyệt đối, chỉ thu field tối thiểu
        if self.emergency and self.category and cat.CATEGORIES[self.category]["can_emergency"]:
            crit = cat.CATEGORIES[self.category]["critical_fields"]
            missing_crit = [k for k in crit if not self.has(k)]
            return Action(ActionType.EMERGENCY,
                          field_key=missing_crit[0] if missing_crit else None,
                          note="Cung cấp hotline cứu hộ, trấn an, chỉ hỏi field tối thiểu để điều xe.")

        # 2. Stuck 2+ lượt -> chuyển human
        if self.fail_count >= MAX_FAIL:
            return Action(ActionType.TRANSFER_HUMAN, note="Đã thử nhiều lần chưa được, đề nghị nối máy nhân viên.")

        # 3. Đang chờ khách xác nhận 1 giá trị (garbled input)
        if self.pending_confirm:
            return Action(ActionType.CONFIRM_FIELD, field_key=self.pending_confirm, value=self.pending_value)

        # 4. Chưa xác định category -> hỏi làm rõ (KHÔNG đoán)
        if self.category is None:
            return Action(ActionType.CLARIFY_INTENT)

        # 5. Còn field verify chưa confirmed -> xác nhận trước
        for k in cat.required_keys(self.category):
            s = self.slots.get(k)
            f = cat.get_field(self.category, k)
            if s and s.value and not s.confirmed and f and f.verify:
                self.pending_confirm, self.pending_value = k, s.value
                return Action(ActionType.CONFIRM_FIELD, field_key=k, value=s.value)

        # 6. Còn field thiếu -> hỏi field đầu tiên còn thiếu (KHÔNG hỏi lại cái đã có)
        missing = self.missing_required()
        if missing:
            return Action(ActionType.ASK_FIELD, field_key=missing[0])

        # 7. Đủ field -> chốt
        return Action(ActionType.COMPLETE)

    # ---------- xuất JSON cuối cuộc gọi ----------
    def to_output(self, post_call: dict = None) -> dict:
        """
        Xuất JSON theo format brief. Field chưa thu -> null (xử lý ca cúp máy giữa chừng).
        post_call: dict {short_summary, sentimental_analysis, emergency} sinh từ LLM ở cuối.
        """
        fields = {}
        if self.category:
            for f in cat.get_fields(self.category):
                s = self.slots.get(f.key)
                fields[f.key] = s.value if (s and s.confirmed) else None
        return {
            "category": self.category,
            "fields": fields,
            "post_call": post_call or {
                "short_summary": None,
                "sentimental_analysis": None,
                "emergency": "yes" if self.emergency else "no",
            },
            "_meta": {  # tiện cho eval/debug, có thể bỏ khi nộp
                "status": self.status.value,
                "turns": self.turn_count,
            },
        }

Writing dialogue_state.py


In [5]:
%%writefile llm_engine.py
"""
Lớp LLM = ghép Qwen2.5 vào state machine để thành callbot biết nói.

QUYẾT ĐỊNH THIẾT KẾ (defend được trong report):
  - EXTRACT dùng LLM (JSON mode), 1 lần/lượt: hiểu khách nói gì -> category + field + cờ.
  - RENDER dùng TEMPLATE xác định (không gọi LLM): nhanh, rẻ, kiểm soát hoàn toàn.
    -> Trên T4 mỗi lượt chỉ 1 lần gọi LLM => latency thấp hơn ~2x, ít điểm fail hơn.
  - POST_CALL dùng LLM 1 lần ở cuối (sinh summary/sentiment/emergency từ transcript).

Vòng lặp 1 lượt (run_turn):
  user_text -> extract(LLM) -> cập nhật state -> next_action(state machine) -> render(template) -> reply
"""
import json
import re
import categories as cat
from dialogue_state import DialogueState, Action, ActionType, Status

RESCUE_HOTLINE = "1900 23 23 89"  # hotline cứu hộ VinFast (thay bằng số chính xác khi nộp)


# ============================================================
# 1) LOAD LLM — Qwen2.5-7B-Instruct 4-bit (fit Colab T4 16GB)
# ============================================================
class LLM:
    def __init__(self, model_id="Qwen/Qwen2.5-7B-Instruct"):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb, device_map="auto"
        )

    def chat(self, system: str, user: str, max_new_tokens=256, temperature=0.1) -> str:
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = self.tok(text, return_tensors="pt").to(self.model.device)
        out = self.model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=temperature > 0, temperature=temperature,
            pad_token_id=self.tok.eos_token_id,
        )
        gen = out[0][inputs["input_ids"].shape[-1]:]
        return self.tok.decode(gen, skip_special_tokens=True).strip()


def _parse_json(text: str) -> dict:
    """Bóc JSON khỏi output LLM (gỡ ```json fences, lấy block {...} đầu tiên)."""
    text = re.sub(r"```(?:json)?", "", text).strip()
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return {}
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return {}


# ============================================================
# 2) EXTRACT — hiểu lượt nói của khách (lần gọi LLM duy nhất/lượt)
# ============================================================
def _category_menu() -> str:
    return "\n".join(f"  {k}: {v['name']} — {v['goal']}" for k, v in cat.CATEGORIES.items())

def _fields_context(state: DialogueState) -> str:
    if not state.category:
        return "(chưa xác định loại cuộc gọi)"
    lines = []
    for f in cat.get_fields(state.category):
        s = state.slots.get(f.key)
        status = "ĐÃ CÓ" if state.has(f.key) else ("chờ xác nhận" if (s and s.value) else "CHƯA CÓ")
        req = "" if f.required else " (không bắt buộc)"
        lines.append(f"  - {f.key} [{f.label}]{req}: {status}")
    return "\n".join(lines)

EXTRACT_SYSTEM = """Bạn là bộ trích xuất thông tin cho tổng đài VinFast. \
Đọc câu KHÁCH HÀNG nói và trả về DUY NHẤT một object JSON, không giải thích, không markdown.

Schema JSON:
{
  "category": "G_1".."G_5" hoặc null,   // chỉ điền khi đủ chắc; mơ hồ -> null
  "updates": { "field_key": "giá trị" },  // thông tin mới khách cung cấp
  "corrections": { "field_key": "giá trị mới" },  // khi khách SỬA thông tin đã nói
  "confirm_response": "yes"|"no"|null,   // khi bot vừa hỏi xác nhận, khách đáp đúng/sai
  "out_of_scope": true|false,            // câu hỏi nằm ngoài 5 loại dịch vụ trên
  "emergency": true|false                // có nguy hiểm tính mạng / cần cứu hộ khẩn cấp
}
Quy tắc: chỉ trích thông tin khách THỰC SỰ nói. Không bịa. Không suy diễn category khi mơ hồ."""

def extract(user_text: str, state: DialogueState, llm: LLM) -> dict:
    user = (
        f"CÁC LOẠI CUỘC GỌI:\n{_category_menu()}\n\n"
        f"LOẠI HIỆN TẠI: {state.category or 'chưa rõ'}\n"
        f"TRẠNG THÁI FIELD:\n{_fields_context(state)}\n"
        f"BOT ĐANG CHỜ XÁC NHẬN: {state.pending_confirm or 'không'}\n\n"
        f"KHÁCH HÀNG NÓI: \"{user_text}\"\n\n"
        f"Trả về JSON:"
    )
    raw = llm.chat(EXTRACT_SYSTEM, user, max_new_tokens=300)
    data = _parse_json(raw)
    # chuẩn hoá để run_turn dùng an toàn
    return {
        "category": data.get("category") if data.get("category") in cat.CATEGORIES else None,
        "updates": data.get("updates") or {},
        "corrections": data.get("corrections") or {},
        "confirm_response": data.get("confirm_response"),
        "out_of_scope": bool(data.get("out_of_scope")),
        "emergency": bool(data.get("emergency")),
    }


# ============================================================
# 3) RENDER — biến Action -> câu tiếng Việt (template, không gọi LLM)
# ============================================================
def render(action: Action, state: DialogueState) -> str:
    t = action.type
    if t == ActionType.CLARIFY_INTENT:
        return ("Dạ VinFast xin nghe ạ. Anh/chị cần hỗ trợ về vấn đề gì để em kết nối đúng bộ phận? "
                "(cứu hộ, bảo hành/sửa chữa, đơn hàng, xe máy điện, hay hỗ trợ kỹ thuật từ xa ạ?)")

    if t == ActionType.ASK_FIELD:
        f = cat.get_field(state.category, action.field_key)
        return f.ask if f else "Anh/chị cho em xin thêm thông tin ạ?"

    if t == ActionType.CONFIRM_FIELD:
        f = cat.get_field(state.category, action.field_key)
        label = f.label if f else "thông tin"
        return f"Em xác nhận lại {label}: \"{action.value}\" — đúng không ạ?"

    if t == ActionType.EMERGENCY:
        msg = (f"Dạ em hiểu tình huống của anh/chị đang khẩn cấp. Trước tiên anh/chị giữ an toàn, "
               f"bật đèn cảnh báo và gọi ngay hotline cứu hộ VinFast {RESCUE_HOTLINE} nếu cần gấp ạ. ")
        if action.field_key:
            f = cat.get_field(state.category, action.field_key)
            msg += f"Để em điều xe hỗ trợ ngay, anh/chị cho em biết {f.label.lower()} ạ?"
        else:
            msg += "Em đang điều phối hỗ trợ tới vị trí của anh/chị ngay ạ."
        return msg

    if t == ActionType.OUT_OF_SCOPE:
        return ("Dạ nội dung này nằm ngoài phạm vi hỗ trợ của tổng đài ạ. Em có thể giúp anh/chị về "
                "cứu hộ, bảo hành, đơn hàng hoặc hỗ trợ kỹ thuật. Hoặc anh/chị có muốn em nối máy "
                "tới nhân viên không ạ?")

    if t == ActionType.TRANSFER_HUMAN:
        state.status = Status.TRANSFERRED
        return ("Dạ để hỗ trợ anh/chị tốt nhất, em xin phép nối máy tới nhân viên tư vấn ạ. "
                "Anh/chị vui lòng giữ máy trong giây lát.")

    if t == ActionType.COMPLETE:
        state.status = Status.COMPLETED
        return ("Dạ em đã ghi nhận đầy đủ thông tin và sẽ xử lý/chuyển bộ phận liên quan ngay ạ. "
                "Cảm ơn anh/chị đã liên hệ VinFast, chúc anh/chị một ngày tốt lành!")

    return "Dạ anh/chị cho em xin thêm thông tin ạ?"


# ============================================================
# 4) RUN_TURN — vòng lặp 1 lượt, nối tất cả lại
# ============================================================
def run_turn(state: DialogueState, user_text: str, llm: LLM):
    """Xử lý 1 lượt khách nói. Trả về (reply, action)."""
    state.log("user", user_text)
    state.turn_count += 1

    ext = extract(user_text, state, llm)
    progressed = False

    # a) đang chờ xác nhận -> xử lý yes/no
    if state.pending_confirm and ext["confirm_response"] in ("yes", "no"):
        key = state.pending_confirm
        state.confirm(key, ext["confirm_response"] == "yes")
        progressed = True

    # b) route category nếu chưa có
    if ext["category"] and not state.category:
        state.set_category(ext["category"])
        progressed = True

    # c) khách sửa thông tin
    for k, v in ext["corrections"].items():
        state.correct(k, v)
        progressed = True

    # d) thông tin mới
    if ext["updates"]:
        need = state.apply_updates(ext["updates"])
        if ext["updates"]:
            progressed = True

    # e) cờ khẩn cấp
    if ext["emergency"] and state.category and cat.CATEGORIES[state.category]["can_emergency"]:
        state.emergency = True

    # f) đếm fail (stuck detection): lượt không tiến triển + không phải out-of-scope hợp lệ
    if progressed:
        state.register_progress()
    elif not ext["out_of_scope"]:
        state.register_fail()

    # g) quyết định hành động (state machine cầm lái)
    action = state.next_action()
    # out-of-scope chỉ override khi không có ưu tiên cao hơn (emergency/transfer)
    if ext["out_of_scope"] and action.type not in (ActionType.EMERGENCY, ActionType.TRANSFER_HUMAN):
        action = Action(ActionType.OUT_OF_SCOPE)

    reply = render(action, state)
    state.log("bot", reply)
    return reply, action


# ============================================================
# 5) POST_CALL — sinh summary/sentiment/emergency từ transcript (LLM, 1 lần cuối)
# ============================================================
POST_CALL_SYSTEM = """Bạn tổng kết một cuộc gọi tổng đài. Đọc toàn bộ hội thoại và trả về DUY NHẤT JSON:
{
  "short_summary": "1-2 câu tóm tắt cuộc gọi và hành động đã thực hiện",
  "sentimental_analysis": "calm" | "frustrated" | "urgent" | ...,  // cảm xúc khách xuyên suốt
  "emergency": "yes" | "no"   // có tình huống nguy hiểm tính mạng/cứu hộ khẩn cấp không
}
Không giải thích, không markdown."""

def generate_post_call(state: DialogueState, llm: LLM) -> dict:
    convo = "\n".join(f"{'Khách' if m['role']=='user' else 'Bot'}: {m['text']}" for m in state.transcript)
    raw = llm.chat(POST_CALL_SYSTEM, f"HỘI THOẠI:\n{convo}\n\nTrả về JSON:", max_new_tokens=200)
    data = _parse_json(raw)
    return {
        "short_summary": data.get("short_summary"),
        "sentimental_analysis": data.get("sentimental_analysis"),
        "emergency": data.get("emergency") or ("yes" if state.emergency else "no"),
    }


def finalize(state: DialogueState, llm: LLM) -> dict:
    """Gọi cuối cuộc gọi (kể cả khi cúp máy giữa chừng) -> JSON output đầy đủ."""
    post = generate_post_call(state, llm)
    return state.to_output(post_call=post)

Writing llm_engine.py


In [6]:
import llm_engine as eng
from dialogue_state import DialogueState

llm = eng.LLM()              # load Qwen2.5-7B-4bit (~5-6GB, fit T4)
state = DialogueState()

# Vòng lặp text-mode để test trước khi gắn mic
while state.status.value == "active":
    text = input("Khách: ")
    if text.lower() in ("quit", "cúp máy"):
        state.mark_hung_up()
        break
    reply, _ = eng.run_turn(state, text, llm)
    print("Bot:", reply)

import json
print(json.dumps(eng.finalize(state, llm), ensure_ascii=False, indent=2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Khách: tôi muốn mua xe ô tô ai tư vấn được không


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Bot: Dạ nội dung này nằm ngoài phạm vi hỗ trợ của tổng đài ạ. Em có thể giúp anh/chị về cứu hộ, bảo hành, đơn hàng hoặc hỗ trợ kỹ thuật. Hoặc anh/chị có muốn em nối máy tới nhân viên không ạ?
Khách: cúp máy
{
  "category": null,
  "fields": {},
  "post_call": {
    "short_summary": "Khách yêu cầu tư vấn mua xe nhưng tổng đài không hỗ trợ nội dung đó, đề xuất các dịch vụ khác và hỏi khách có muốn chuyển tiếp không.",
    "sentimental_analysis": "calm",
    "emergency": "no"
  },
  "_meta": {
    "status": "active",
    "turns": 1
  }
}
